# Source-release notes

Artifact inspection and visualization.

Run this notebook with its containing directory as the kernel working directory. Experimental inputs and saved outputs are not included in this source release. See the [workflow guide](../docs/WORKFLOW.md), [data requirements](../docs/DATA.md), and [reproducibility notes](../docs/REPRODUCIBILITY.md) before execution.

Scientific settings are preserved. Packaging changes are recorded in [PREPARATION.md](../docs/PREPARATION.md). Original cell numbers in that record do not include this added introductory cell.


In [ ]:
import os
import math
import numpy as np
import cv2
import matplotlib.pyplot as plt

import csv

In [ ]:

# Directory containing synthetic segments
mfm_segments_dir = "{}/synthetic_mfm_segments".format(os.getcwd())
segment_index = 55
segment_path = os.path.join(mfm_segments_dir, f"synthetic_mfm_segment_{segment_index}.png")

# Load and process the segment
segment = cv2.imread(segment_path, cv2.IMREAD_COLOR)
if segment is None:
    print(f"Segment {segment_index} could not be loaded.")
else:
    # Convert the segment to grayscale and create a binary mask
    segment_gray = cv2.cvtColor(segment, cv2.COLOR_BGR2GRAY)
    _, bit_mask = cv2.threshold(segment_gray, 10, 255, cv2.THRESH_BINARY)

    # Find contours and filter by area
    contours, _ = cv2.findContours(bit_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    min_contour_area = 50  # Set minimum contour area threshold
    contours = [cnt for cnt in contours if cv2.contourArea(cnt) > min_contour_area]

    if contours:
        # Fit an ellipse to find the orientation
        if len(contours[0]) >= 5:
            ellipse = cv2.fitEllipse(contours[0])
            center = (int(ellipse[0][0]), int(ellipse[0][1]))
            angle = ellipse[2]

            # Rotate the major axis by 180 degrees for correct scanning direction
            rotated_angle = angle + 180
            brightness_jump_threshold_percentage = 0.8  # Set threshold for sudden brightness jumps

            # Function to scan brightness radially
            def scan_radial_brightness(mask, center, expected_gradient):
                pixels = np.column_stack(np.where(mask > 0))
                if pixels.size == 0:
                    return [], []

                # Calculate radial distances from the center
                distances = np.sqrt((pixels[:, 0] - center[1])**2 + (pixels[:, 1] - center[0])**2)
                sorted_indices = np.argsort(distances)
                sorted_pixels = pixels[sorted_indices]

                # Calculate brightness values and gradients
                brightness_values = np.array([segment_gray[tuple(p)] for p in sorted_pixels], dtype=np.float32)
                brightness_gradients = np.diff(brightness_values)
                brightness_range = brightness_values.max() - brightness_values.min()
                threshold = max(brightness_range * brightness_jump_threshold_percentage, 20)

                artifact_pixels = []
                for i, gradient in enumerate(brightness_gradients):
                    if expected_gradient == "increasing" and gradient < -threshold:
                        artifact_pixels.extend(sorted_pixels[i + 1:].tolist())
                        break  # Stop scanning once a jump is detected
                    elif expected_gradient == "decreasing" and gradient > threshold:
                        artifact_pixels.extend(sorted_pixels[i + 1:].tolist())
                        break  # Stop scanning once a jump is detected

                return sorted_pixels, artifact_pixels

            # Divide segment into two halves
            mask1, mask2 = np.zeros_like(bit_mask), np.zeros_like(bit_mask)
            for i in range(bit_mask.shape[0]):
                for j in range(bit_mask.shape[1]):
                    dx, dy = j - center[0], i - center[1]
                    projection = dx * math.cos(math.radians(rotated_angle)) + dy * math.sin(math.radians(rotated_angle))
                    if projection > 0:
                        mask1[i, j] = bit_mask[i, j]
                    else:
                        mask2[i, j] = bit_mask[i, j]

            # Scan each half radially for artifacts
            sorted_side1, artifacts_side1 = scan_radial_brightness(mask1, center, "increasing")
            sorted_side2, artifacts_side2 = scan_radial_brightness(mask2, center, "decreasing")

            # Combine artifact pixels
            artifact_arrays = [artifacts_side1, artifacts_side2]
            non_empty_artifact_arrays = [arr for arr in artifact_arrays if len(arr) > 0]

            if len(non_empty_artifact_arrays) > 0:
                all_artifact_pixels = np.vstack(non_empty_artifact_arrays)
            else:
                all_artifact_pixels = np.empty((0, 2), dtype=int)  # Create an empty array with shape (0, 2)

            # Create an artifact mask for contours
            artifact_mask = np.zeros_like(bit_mask)
            for p in all_artifact_pixels:
                artifact_mask[p[0], p[1]] = 255

            # Find contours for artifact regions
            artifact_contours, _ = cv2.findContours(artifact_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            # Draw outlines of the artifact regions on the original segment
            artifact_highlight = segment.copy()
            cv2.drawContours(artifact_highlight, artifact_contours, -1, (0, 255, 255), 1)  # Yellow outline

            plt.figure(figsize=(8, 8))
            plt.imshow(cv2.cvtColor(artifact_highlight, cv2.COLOR_BGR2RGB))
            plt.title(f"Segment {segment_index}: Artifact Regions Outlined")
            plt.axis('off')
            if len(all_artifact_pixels) == 0:
                plt.text(10, 10, "No artifacts detected", color='red', fontsize=12)
            plt.show()

            # Create a mask for the artifact-free region
            artifact_free_mask = np.ones_like(segment_gray, dtype=np.uint8) * 255
            for p in all_artifact_pixels:
                artifact_free_mask[p[0], p[1]] = 0

            artifact_free_segment = cv2.bitwise_and(segment, segment, mask=artifact_free_mask)

            # Plot the second image with artifact-free segment
            plt.figure(figsize=(8, 8))
            plt.imshow(cv2.cvtColor(artifact_free_segment, cv2.COLOR_BGR2RGB))
            plt.title(f"Segment {segment_index}: Artifact-Free Region")
            plt.axis('off')
            plt.show()
    else:
        print(f"No contours found for segment {segment_index}.")


In [ ]:


# Directory containing synthetic segments
mfm_segments_dir = "{}/synthetic_mfm_segments".format(os.getcwd())
segment_index = 55
segment_path = os.path.join(mfm_segments_dir, f"synthetic_mfm_segment_{segment_index}.png")

# Load and process the segment
segment = cv2.imread(segment_path, cv2.IMREAD_COLOR)
if segment is None:
    print(f"Segment {segment_index} could not be loaded.")
else:
    # Convert the segment to grayscale and create a binary mask
    segment_gray = cv2.cvtColor(segment, cv2.COLOR_BGR2GRAY)
    _, bit_mask = cv2.threshold(segment_gray, 10, 255, cv2.THRESH_BINARY)

    # Find contours and filter by area
    contours, _ = cv2.findContours(bit_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    min_contour_area = 50  # Set minimum contour area threshold
    contours = [cnt for cnt in contours if cv2.contourArea(cnt) > min_contour_area]

    if contours:
        # Fit an ellipse to find the orientation
        if len(contours[0]) >= 5:
            ellipse = cv2.fitEllipse(contours[0])
            center = (int(ellipse[0][0]), int(ellipse[0][1]))
            angle = ellipse[2]

            # Rotate the major axis by 180 degrees for correct scanning direction
            rotated_angle = angle + 180
            brightness_jump_threshold_percentage = 0.8  # Set threshold for sudden brightness jumps

            # Function to scan brightness radially
            def scan_radial_brightness(mask, center, expected_gradient):
                pixels = np.column_stack(np.where(mask > 0))
                if pixels.size == 0:
                    return [], []

                # Calculate radial distances from the center
                distances = np.sqrt((pixels[:, 0] - center[1])**2 + (pixels[:, 1] - center[0])**2)
                sorted_indices = np.argsort(distances)
                sorted_pixels = pixels[sorted_indices]

                # Calculate brightness values and gradients
                brightness_values = np.array([segment_gray[tuple(p)] for p in sorted_pixels], dtype=np.float32)
                brightness_gradients = np.diff(brightness_values)
                brightness_range = brightness_values.max() - brightness_values.min()
                threshold = max(brightness_range * brightness_jump_threshold_percentage, 20)

                artifact_pixels = []
                for i, gradient in enumerate(brightness_gradients):
                    if expected_gradient == "increasing" and gradient < -threshold:
                        artifact_pixels.extend(sorted_pixels[i + 1:].tolist())
                        break  # Stop scanning once a jump is detected
                    elif expected_gradient == "decreasing" and gradient > threshold:
                        artifact_pixels.extend(sorted_pixels[i + 1:].tolist())
                        break  # Stop scanning once a jump is detected

                return sorted_pixels, artifact_pixels

            # Divide segment into two halves
            mask1, mask2 = np.zeros_like(bit_mask), np.zeros_like(bit_mask)
            for i in range(bit_mask.shape[0]):
                for j in range(bit_mask.shape[1]):
                    dx, dy = j - center[0], i - center[1]
                    projection = dx * math.cos(math.radians(rotated_angle)) + dy * math.sin(math.radians(rotated_angle))
                    if projection > 0:
                        mask1[i, j] = bit_mask[i, j]
                    else:
                        mask2[i, j] = bit_mask[i, j]

            # Scan each half radially for artifacts
            _, artifacts_side1 = scan_radial_brightness(mask1, center, "increasing")
            _, artifacts_side2 = scan_radial_brightness(mask2, center, "decreasing")

            # Combine artifact pixels
            artifact_arrays = [artifacts_side1, artifacts_side2]
            non_empty_artifact_arrays = [arr for arr in artifact_arrays if len(arr) > 0]

            if len(non_empty_artifact_arrays) > 0:
                all_artifact_pixels = np.vstack(non_empty_artifact_arrays)
            else:
                all_artifact_pixels = np.empty((0, 2), dtype=int)  # Create an empty array with shape (0, 2)

            # Create a mask for the artifact-free region
            artifact_free_mask = np.zeros_like(bit_mask)
            artifact_free_mask[bit_mask > 0] = 255  # Initialize with original segment
            for p in all_artifact_pixels:
                artifact_free_mask[p[0], p[1]] = 0  # Remove artifact pixels

            # Recalculate contours for artifact-free region
            artifact_free_contours, _ = cv2.findContours(artifact_free_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if artifact_free_contours:
                x, y, w, h = cv2.boundingRect(artifact_free_contours[0])
                bit_diagonal_length = int(np.sqrt(w**2 + h**2))
                max_arrow_length = int(0.90 * bit_diagonal_length)
                min_arrow_length = int(0.55 * bit_diagonal_length)
                print(f"Artifact-Free Bounding Box (x={x}, y={y}, w={w}, h={h}), Diagonal length: {bit_diagonal_length}")
            else:
                print("No valid artifact-free contours detected.")
                bit_diagonal_length = 0
                max_arrow_length = 0
                min_arrow_length = 0

            # Calculate brightness for each side
            artifact_free_pixels_side1 = np.column_stack(np.where(mask1 > 0))
            artifact_free_pixels_side2 = np.column_stack(np.where(mask2 > 0))
            artifact_free_pixels_side1 = [p for p in artifact_free_pixels_side1 if tuple(p) not in all_artifact_pixels]
            artifact_free_pixels_side2 = [p for p in artifact_free_pixels_side2 if tuple(p) not in all_artifact_pixels]

            brightness_side1 = np.nanmean([segment_gray[tuple(p)] for p in artifact_free_pixels_side1]) if artifact_free_pixels_side1 else 0
            brightness_side2 = np.nanmean([segment_gray[tuple(p)] for p in artifact_free_pixels_side2]) if artifact_free_pixels_side2 else 0

            # Determine the brighter side and calculate arrow length
            is_side1_brighter = brightness_side1 > brightness_side2
            brightness_difference = abs(brightness_side1 - brightness_side2)
            normalized_strength = brightness_difference / 255.0
            arrow_length = max(min_arrow_length, min_arrow_length + int(normalized_strength * (max_arrow_length - min_arrow_length)))

            # Calculate arrow direction based on brightness
            angle_adjusted = rotated_angle if is_side1_brighter else rotated_angle + 270
            dx_brightest = int((arrow_length / 2) * math.cos(math.radians(angle_adjusted)))
            dy_brightest = int((arrow_length / 2) * math.sin(math.radians(angle_adjusted)))

            # Define and draw arrow on artifact-free segment
            start_point = (center[0] - dx_brightest, center[1] - dy_brightest)
            end_point = (center[0] + dx_brightest, center[1] + dy_brightest)
            cv2.arrowedLine(segment, start_point, end_point, (255, 255, 255), 2, tipLength=0.2)

            # Plot the artifact-free segment with arrow
            artifact_free_segment = cv2.bitwise_and(segment, segment, mask=artifact_free_mask)
            plt.figure(figsize=(8, 8))
            plt.imshow(cv2.cvtColor(artifact_free_segment, cv2.COLOR_BGR2RGB))
            plt.title(f"Segment {segment_index}: Artifact-Free Region with Magnetic Dipole")
            plt.axis('off')
            plt.show()
    else:
        print(f"No contours found for segment {segment_index}.")

In [ ]:
import os
import math
import numpy as np
import cv2
import csv
import matplotlib.pyplot as plt

# Directories for synthetic segments and output
mfm_segments_dir = "{}/synthetic_mfm_segments".format(os.getcwd())
output_dir = "{}/synthetic_artifact_free_mfm_segments_with_arrows".format(os.getcwd())
os.makedirs(output_dir, exist_ok=True)

# CSV file to save direction and strength of each segment
csv_path = os.path.join(output_dir, "synthetic_magnetic_moments_without_artifacts.csv")

# Initialize counters
right_arrow_count = 0
left_arrow_count = 0
net_magnetic_moment = 0

# Open CSV file for writing
with open(csv_path, mode='w', newline='') as file:
    csv_writer = csv.writer(file)
    csv_writer.writerow(["Segment", "Direction", "Magnetic_Moment_Strength"])

    # Loop through all segment files
    segment_files = [f for f in os.listdir(mfm_segments_dir) if f.endswith(".png")]
    for segment_file in segment_files:
        segment_index = int(segment_file.split('_')[-1].split('.')[0])
        segment_path = os.path.join(mfm_segments_dir, segment_file)

        # Load and process the segment
        segment = cv2.imread(segment_path, cv2.IMREAD_COLOR)
        if segment is None:
            print(f"Segment {segment_index} could not be loaded.")
            continue

        segment_gray = cv2.cvtColor(segment, cv2.COLOR_BGR2GRAY)
        _, bit_mask = cv2.threshold(segment_gray, 10, 255, cv2.THRESH_BINARY)

        # Find contours and filter by area
        contours, _ = cv2.findContours(bit_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        min_contour_area = 50
        contours = [cnt for cnt in contours if cv2.contourArea(cnt) > min_contour_area]

        if not contours:
            print(f"No contours found for segment {segment_index}.")
            continue

        # Fit an ellipse to find orientation
        largest_contour = max(contours, key=cv2.contourArea)
        if len(largest_contour) < 5:
            print(f"Not enough points for ellipse fitting for segment {segment_index}.")
            continue

        ellipse = cv2.fitEllipse(largest_contour)
        center = (int(ellipse[0][0]), int(ellipse[0][1]))
        angle = ellipse[2]

        # Rotate the major axis by 180 degrees for correct scanning direction
        rotated_angle = angle + 180

        # Function to scan brightness radially
        def scan_radial_brightness(mask, center, expected_gradient):
            pixels = np.column_stack(np.where(mask > 0))
            if pixels.size == 0:
                return [], []

            distances = np.sqrt((pixels[:, 0] - center[1])**2 + (pixels[:, 1] - center[0])**2)
            sorted_indices = np.argsort(distances)
            sorted_pixels = pixels[sorted_indices]

            brightness_values = np.array([segment_gray[tuple(p)] for p in sorted_pixels], dtype=np.float32)
            brightness_gradients = np.diff(brightness_values)
            brightness_range = brightness_values.max() - brightness_values.min()
            threshold = max(brightness_range * 0.8, 20)

            artifact_pixels = []
            for i, gradient in enumerate(brightness_gradients):
                if expected_gradient == "increasing" and gradient < -threshold:
                    artifact_pixels.extend(sorted_pixels[i + 1:].tolist())
                    break
                elif expected_gradient == "decreasing" and gradient > threshold:
                    artifact_pixels.extend(sorted_pixels[i + 1:].tolist())
                    break

            return sorted_pixels, artifact_pixels

        # Divide segment into two halves
        mask1, mask2 = np.zeros_like(bit_mask), np.zeros_like(bit_mask)
        for i in range(bit_mask.shape[0]):
            for j in range(bit_mask.shape[1]):
                dx, dy = j - center[0], i - center[1]
                projection = dx * math.cos(math.radians(rotated_angle)) + dy * math.sin(math.radians(rotated_angle))
                if projection > 0:
                    mask1[i, j] = bit_mask[i, j]
                else:
                    mask2[i, j] = bit_mask[i, j]

        # Scan each half radially for artifacts
        _, artifacts_side1 = scan_radial_brightness(mask1, center, "increasing")
        _, artifacts_side2 = scan_radial_brightness(mask2, center, "decreasing")

        # Combine artifact pixels
        artifact_arrays = [artifacts_side1, artifacts_side2]
        non_empty_artifact_arrays = [arr for arr in artifact_arrays if len(arr) > 0]
        all_artifact_pixels = np.vstack(non_empty_artifact_arrays) if non_empty_artifact_arrays else np.empty((0, 2), dtype=int)

        # Create a mask for the artifact-free region
        artifact_free_mask = np.zeros_like(bit_mask)
        artifact_free_mask[bit_mask > 0] = 255
        for p in all_artifact_pixels:
            artifact_free_mask[p[0], p[1]] = 0

        # Recalculate contours for artifact-free region
        artifact_free_contours, _ = cv2.findContours(artifact_free_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if artifact_free_contours:
            x, y, w, h = cv2.boundingRect(artifact_free_contours[0])
            bit_diagonal_length = int(np.sqrt(w**2 + h**2))
            max_arrow_length = int(0.90 * bit_diagonal_length)
            min_arrow_length = int(0.55 * bit_diagonal_length)
        else:
            print(f"No valid artifact-free contours detected for segment {segment_index}.")
            continue

        # Calculate brightness for each side
        artifact_free_pixels_side1 = np.column_stack(np.where(mask1 > 0))
        artifact_free_pixels_side2 = np.column_stack(np.where(mask2 > 0))
        artifact_free_pixels_side1 = [p for p in artifact_free_pixels_side1 if tuple(p) not in all_artifact_pixels]
        artifact_free_pixels_side2 = [p for p in artifact_free_pixels_side2 if tuple(p) not in all_artifact_pixels]

        brightness_side1 = np.nanmean([segment_gray[tuple(p)] for p in artifact_free_pixels_side1]) if artifact_free_pixels_side1 else 0
        brightness_side2 = np.nanmean([segment_gray[tuple(p)] for p in artifact_free_pixels_side2]) if artifact_free_pixels_side2 else 0

        # Determine the brighter side and calculate arrow length
        is_side1_brighter = brightness_side1 > brightness_side2
        brightness_difference = abs(brightness_side1 - brightness_side2)
        normalized_strength = brightness_difference / 255.0
        arrow_length = max(min_arrow_length, min_arrow_length + int(normalized_strength * (max_arrow_length - min_arrow_length)))

        # Calculate arrow direction based on brightness
        angle_adjusted = rotated_angle if is_side1_brighter else rotated_angle + 270
        dx_brightest = int((arrow_length / 2) * math.cos(math.radians(angle_adjusted)))
        dy_brightest = int((arrow_length / 2) * math.sin(math.radians(angle_adjusted)))

        # Define and draw arrow on artifact-free segment
        start_point = (center[0] - dx_brightest, center[1] - dy_brightest)
        end_point = (center[0] + dx_brightest, center[1] + dy_brightest)
        cv2.arrowedLine(segment, start_point, end_point, (255, 255, 255), 2, tipLength=0.2)

        # Save artifact-free segment with arrow
        artifact_free_segment = cv2.bitwise_and(segment, segment, mask=artifact_free_mask)
        output_path = os.path.join(output_dir, f"synthetic_artifact_free_segment_{segment_index}.png")
        cv2.imwrite(output_path, artifact_free_segment)

        # Write to CSV and update statistics
        direction_label = 1 if is_side1_brighter else 0
        csv_writer.writerow([segment_index, direction_label, arrow_length])
        print(f"Segment {segment_index}: Direction: {'Left' if direction_label == 1 else 'Right'}, Strength: {arrow_length}")
        if direction_label == 1:
            left_arrow_count += 1
        else:
            right_arrow_count += 1
        net_magnetic_moment += arrow_length if direction_label == 0 else -arrow_length

# Print summary statistics
print("\nTotal arrows pointing right:", right_arrow_count)
print("Total arrows pointing left:", left_arrow_count)
print("Net magnetic moment (based on arrow lengths):", net_magnetic_moment)
overall_magnetic_moment = "Right" if net_magnetic_moment > 0 else "Left"
print("The overall magnetic moment direction is:", overall_magnetic_moment)


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Directories for synthetic segments and output
synthetic_segments_dir = "{}/synthetic_artifact_free_mfm_segments_with_arrows".format(os.getcwd())
original_segments_dir = "{}/bit_visualization_segments".format(os.getcwd())
output_dir = "{}/reconstructed_image_without_artifacts".format(os.getcwd())
os.makedirs(output_dir, exist_ok=True)

# Path to the original "Bits Segmentation Visualization" image
segmentation_visualization_path = "bit_segmentation_visualization.png"

# Load the original segmentation visualization to get the original dimensions
segmentation_visualization = cv2.imread(segmentation_visualization_path)
if segmentation_visualization is None:
    print("Segmentation visualization image could not be loaded.")
else:
    # Create a blank canvas for the reconstructed image
    reassembled_image = np.zeros_like(segmentation_visualization)

    # Convert to grayscale and find connected components in the segmentation visualization
    segmentation_gray = cv2.cvtColor(segmentation_visualization, cv2.COLOR_BGR2GRAY)
    _, binary_mask = cv2.threshold(segmentation_gray, 1, 255, cv2.THRESH_BINARY)
    num_labels, labels_im = cv2.connectedComponents(binary_mask)

    # Step 1: Loop over each segment label in the mask
    for label in range(1, num_labels):
        # Paths for synthetic and original segments
        synthetic_segment_path = os.path.join(synthetic_segments_dir, f"synthetic_artifact_free_segment_{label}.png")
        original_segment_path = os.path.join(original_segments_dir, f"segment_{label}.png")

        # Load both synthetic and original segments
        synthetic_segment = cv2.imread(synthetic_segment_path, cv2.IMREAD_COLOR)
        original_segment = cv2.imread(original_segment_path, cv2.IMREAD_COLOR)

        if synthetic_segment is None:
            print(f"Synthetic Segment {label} could not be loaded. Skipping...")
            continue
        if original_segment is None:
            print(f"Original Segment {label} could not be loaded. Skipping...")
            continue

        # Remove the background of the synthetic segment
        synthetic_gray = cv2.cvtColor(synthetic_segment, cv2.COLOR_BGR2GRAY)
        _, synthetic_mask = cv2.threshold(synthetic_gray, 10, 255, cv2.THRESH_BINARY)
        x_s, y_s, w_s, h_s = cv2.boundingRect(synthetic_mask)
        cropped_synthetic = synthetic_segment[y_s:y_s + h_s, x_s:x_s + w_s]

        # Find the bounding box of the segment in the original mask
        mask = (labels_im == label).astype(np.uint8) * 255
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if not contours:
            print(f"No contours found for segment {label}. Skipping...")
            continue

        # Get the bounding box of the current segment area in the mask
        x, y, w, h = cv2.boundingRect(contours[0])

        # Resize the cropped synthetic segment to match the bounding box of the original segment
        synthetic_resized = cv2.resize(cropped_synthetic, (w, h))

        # Overlay the synthetic segment onto the reassembled image
        reassembled_image[y:y + h, x:x + w] = synthetic_resized

        # # Add the label to the segment
        # label_position = (x + w // 2, y + h // 2)  # Place label at the center of the bounding box
        # cv2.putText(
        #     reassembled_image,
        #     str(label),
        #     label_position,
        #     cv2.FONT_HERSHEY_SIMPLEX,
        #     0.25,  # Font scale
        #     (0, 0, 255),  # Text color (red)
        #     1,  # Thickness
        #     cv2.LINE_AA
        # )

    # Step 2: Save and display the reassembled image
    output_path = os.path.join(output_dir, "reconstructed_image_with_labels.png")
    cv2.imwrite(output_path, reassembled_image)

    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(reassembled_image, cv2.COLOR_BGR2RGB))
    plt.title("Reconstructed MFM Image from Synthetic Segments with Arrows and Labels")
    plt.axis('off')
    plt.show()

    print(f"Reconstructed image saved to {output_path}")
